# sol05: Concurrent Web Crawler

Contains:
- the same scenario as `05_mock`
- one complete reference implementation
- grading tests


In [ ]:
from collections import deque
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from copy import deepcopy
from threading import Lock
from typing import Any
from urllib.parse import urldefrag, urlparse
import time

WEB_GRAPH = {
    "https://docs.local/start": [
        "https://docs.local/a#intro",
        "https://docs.local/b",
        "https://external.com/ignore",
    ],
    "https://docs.local/a": [
        "https://docs.local/b",
        "https://docs.local/c",
    ],
    "https://docs.local/b": [
        "https://docs.local/c#part",
        "https://docs.local/d",
    ],
    "https://docs.local/c": [
        "https://docs.local/start",
    ],
    "https://docs.local/d": [],
}


class FakeHtmlParser:
    def __init__(self, graph: dict[str, list[str]], delay_seconds: float = 0.0) -> None:
        self._graph = deepcopy(graph)
        self._delay_seconds = delay_seconds
        self._calls: list[str] = []
        self._lock = Lock()

    def getUrls(self, url: str) -> list[str]:
        if self._delay_seconds:
            time.sleep(self._delay_seconds)
        with self._lock:
            self._calls.append(url)
        return deepcopy(self._graph.get(url, []))

    def call_count(self) -> int:
        with self._lock:
            return len(self._calls)


In [ ]:
def normalize_url(url: str) -> str:
    clean, _ = urldefrag(url)
    parsed = urlparse(clean)
    if clean.endswith("/") and parsed.path not in ("", "/"):
        return clean[:-1]
    return clean


def crawl_single_thread(start_url: str, parser: Any) -> list[str]:
    start = normalize_url(start_url)
    host = urlparse(start).hostname
    visited: set[str] = {start}
    queue: deque[str] = deque([start])

    while queue:
        current = queue.popleft()
        for nxt in parser.getUrls(current):
            url = normalize_url(nxt)
            if urlparse(url).hostname != host:
                continue
            if url in visited:
                continue
            visited.add(url)
            queue.append(url)

    return sorted(visited)


def crawl_multi_thread(start_url: str, parser: Any, max_workers: int = 4) -> list[str]:
    start = normalize_url(start_url)
    host = urlparse(start).hostname
    visited: set[str] = {start}
    lock = Lock()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        in_flight: dict[Any, str] = {executor.submit(parser.getUrls, start): start}

        while in_flight:
            done, _ = wait(set(in_flight), return_when=FIRST_COMPLETED)
            for future in done:
                in_flight.pop(future)
                for nxt in future.result():
                    url = normalize_url(nxt)
                    if urlparse(url).hostname != host:
                        continue
                    with lock:
                        if url in visited:
                            continue
                        visited.add(url)
                    in_flight[executor.submit(parser.getUrls, url)] = url

    return sorted(visited)


In [ ]:
def run_exam05_tests() -> None:
    expected = [
        "https://docs.local/a",
        "https://docs.local/b",
        "https://docs.local/c",
        "https://docs.local/d",
        "https://docs.local/start",
    ]

    parser_single = FakeHtmlParser(WEB_GRAPH, delay_seconds=0.0)
    single = crawl_single_thread("https://docs.local/start#home", parser_single)
    assert single == expected
    assert parser_single.call_count() == len(expected)

    parser_multi = FakeHtmlParser(WEB_GRAPH, delay_seconds=0.01)
    multi = crawl_multi_thread("https://docs.local/start#home", parser_multi, max_workers=4)
    assert multi == expected
    assert parser_multi.call_count() == len(expected)

    assert single == multi
    print("05_mock tests passed")


run_exam05_tests()


## Walkthrough: Exactly How to Solve `05_mock`

### 0) First 2 minutes
- Decide canonical URL format first (`urldefrag` + parse host).
- Decide dedupe rule: add to visited when enqueued/scheduled.

### 1) Should I read tests now?
Yes:
- Expected output list shows exactly which URLs survive.
- Both single and multi must return same sorted result.
- Parser call count must equal number of visited URLs.

### 2) Coding order
1. `normalize_url` first (easy win).
2. `crawl_single_thread` using queue + visited + same-host filter.
3. `crawl_multi_thread` with executor + in-flight futures + lock-protected visited updates.

### 3) One concrete example to narrate aloud
From `start`, parser returns:
- `https://docs.local/a#intro` -> normalize to `/a`
- `https://external.com/ignore` -> filtered by host check
This demonstrates why normalization and same-host filtering happen before scheduling.

### 4) What to say while coding
- "I am using single-thread as correctness baseline before concurrency."
- "I protect visited-set updates to avoid duplicate scheduling races."
- "I normalize URLs before dedupe, otherwise fragments would create false duplicates."

### 5) Self-check before final run
- Is each URL fetched at most once?
- Can external host URLs leak in?
- Do multi-thread and single-thread outputs match exactly?
